In [1]:
from pathlib import Path
import pandas as pd

workspace_root = (Path.cwd() / '..').resolve()

# Target JSON (for this notebook context)
json_path = workspace_root / 'JSON Whole Model' / 'Ifc2x3_Duplex_Architecture.json'
assert json_path.exists(), f'JSON not found: {json_path}'

# Read Excel from COBie folder (columns G and I)
excel_path = workspace_root / 'COBie' / 'Uniclass2015_EF_v1_16.xlsx'
assert excel_path.exists(), f'Excel not found: {excel_path}'

# Read only columns G and I
df = pd.read_excel(excel_path, usecols='G,I')
df.columns = ['G', 'I']

# Find rows where Column G == 'Walls'
walls_rows = df[df['G'].astype(str).str.strip().str.casefold() == 'walls']

if walls_rows.empty:
    print("No match found for 'Walls' in column G.")
else:
    first_value = walls_rows['I'].iloc[0]
    print('First corresponding value in column I for Walls:')
    print(first_value)

First corresponding value in column I for Walls:
EF_25_10 : Walls


In [2]:
import json
import pandas as pd

# Load JSON Whole Model data
with json_path.open('r', encoding='utf-8') as f:
    model_data = json.load(f)

def extract_guid(properties):
    if not isinstance(properties, list):
        return None
    for prop in properties:
        if not isinstance(prop, dict):
            continue
        if str(prop.get('displayName', '')).strip().lower() == 'guid':
            guid_value = prop.get('value')
            return str(guid_value).strip() if guid_value is not None else None
    return None

def find_wall_property_matches(properties):
    matches = []
    if not isinstance(properties, list):
        return matches

    for prop in properties:
        if not isinstance(prop, dict):
            continue
        haystack_parts = [
            str(prop.get('category', '')),
            str(prop.get('displayName', '')),
            str(prop.get('value', ''))
        ]
        haystack = ' | '.join(haystack_parts).lower()
        if 'wall' in haystack:
            matches.append(prop)
    return matches

matched_rows = []
for item in model_data:
    if not isinstance(item, dict):
        continue
    properties = item.get('Properties')
    wall_matches = find_wall_property_matches(properties)
    if not wall_matches:
        continue

    matched_rows.append({
        'Name': item.get('Name'),
        'DbId': item.get('DbId'),
        'GUID': extract_guid(properties),
        'WallPropertyMatchCount': len(wall_matches)
    })

matched_df = pd.DataFrame(matched_rows)

print(f"Total objects with 'Wall' in Properties: {len(matched_df)}")

if matched_df.empty:
    print("No objects found with 'Wall' in Properties.")
else:
    # Detail table with GUID (up to 200 rows)
    detail_table = (
        matched_df[['Name', 'DbId', 'GUID', 'WallPropertyMatchCount']]
        .sort_values(['Name', 'DbId'], kind='stable')
        .reset_index(drop=True)
    )
    print("\nDetail table (objects containing 'Wall' in Properties):")
    display(detail_table.head(200))

    # Summary table: count by Name
    summary_table = (
        matched_df.groupby('Name', dropna=False)
        .agg(
            ObjectCount=('DbId', 'count'),
            GuidCount=('GUID', lambda s: s.dropna().nunique())
        )
        .reset_index()
        .sort_values(['ObjectCount', 'Name'], ascending=[False, True], kind='stable')
        .reset_index(drop=True)
    )
    print("\nSummary table (count by Name):")
    display(summary_table.head(200))

Total objects with 'Wall' in Properties: 231

Detail table (objects containing 'Wall' in Properties):


,Name,DbId,GUID,WallPropertyMatchCount
0,A102,9,0b74b3fa-1a92-405e-9ac9-d59067be1d42,2
1,A103,10,0b74b3fa-1a92-405e-9ac9-d59067be1d7f,2
2,A104,11,0b74b3fa-1a92-405e-9ac9-d59067be1d78,2
3,A202,677,0b74b3fa-1a92-405e-9ac9-d59067be1d66,2
4,A203,676,0b74b3fa-1a92-405e-9ac9-d59067be1d65,2
...,...,...,...,...
195,Plan,539,None,1
196,Plan,567,None,1
197,Plan,578,None,1
198,Plan,958,None,1



Summary table (count by Name):


,Name,ObjectCount,GuidCount
0,Body,64,0
1,Plan,46,0
2,A102,1,1
3,A103,1,1
4,A104,1,1
...,...,...,...
118,Wall Foundation:Bearing Footing - 900 x 300:18...,1,1
119,Wall Foundation:Bearing Footing - 900 x 300:18...,1,1
120,Wall Foundation:Bearing Footing - 900 x 300:18...,1,1
121,Wall Foundation:Bearing Footing - 900 x 300:18...,1,1
